# Chapter 6 &mdash; Minimization as a Fixed-Point Computation

**Concept 10 of the Chapter 6 decomposition:** *Minimization as a Fixed-Point Computation: `fixptDist`*

Apply the propagation rule until the table stops changing &mdash; a fixed point, reached monotonically.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Fixed-Point-Minimization/Concept-Fixed-Point-Minimization.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


`fixptDist` is the whole algorithm in one idea: **apply a rule repeatedly until
nothing changes**.

The rule only ever turns $-1$ into a number, never the other way, so the table is
**monotone**; the table is **finite**; therefore a **fixed point** exists and is
reached in finitely many rounds &mdash; at most $|Q|-1$ of them.

This is the same pattern as dataflow analysis in compilers and as the closure
computations in Chapter 11. Recognising it saves you from re-deriving termination
arguments.

## 2. Definitions

### A chain machine, which needs the maximum number of rounds

In [ ]:
chain = md2mc('''DFA
I  : 0 -> A
A  : 0 -> B
B  : 0 -> C
C  : 0 -> F
F  : 0 -> F
I  : 1 -> I
A  : 1 -> A
B  : 1 -> B
C  : 1 -> C
F  : 1 -> F
''')

### The fixed-point loop, instrumented to count rounds

In [ ]:
def fixpoint(D):
    qs = sorted(D["Q"])
    pairs = [(a, b) for i, a in enumerate(qs) for b in qs[i+1:]]
    dist = {p: (0 if (p[0] in D["F"]) != (p[1] in D["F"]) else -1) for p in pairs}
    def key(x, y): return (x, y) if (x, y) in dist else (y, x)
    rounds, k = 0, 0
    while True:
        changed = False
        for (p, q) in pairs:
            if dist[(p, q)] != -1: continue
            for a in sorted(D["Sigma"]):
                s, t = step_dfa(D, p, a), step_dfa(D, q, a)
                if s != t and dist[key(s, t)] != -1 and dist[key(s, t)] <= k:
                    dist[(p, q)] = k + 1; changed = True; break
        rounds += 1; k += 1
        if not changed: return dist, rounds

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;9.&nbsp;Merging Equivalence Classes into the Minimal Machine](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Merging-Equivalence-Classes/Concept-Merging-Equivalence-Classes.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6-DFAOps/README.md) &nbsp;&middot;&nbsp; [Ch6&nbsp;11.&nbsp;Worked Example: Union, Minimization, and the Two Comparison Predicates](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Worked-Union-Minimization/Concept-Worked-Union-Minimization.ipynb)&nbsp;&rarr;

---

## 3. Tests

Monotonicity: marks are only ever added, never removed.

In [ ]:
dist, rounds = fixpoint(chain)
print("rounds to fixpoint :", rounds, "   |Q|-1 =", len(chain["Q"]) - 1)
print("distances :", dict(sorted(dist.items())))
assert rounds <= len(chain["Q"])

Every pair in this chain is distinguishable, so nothing merges.

In [ ]:
print("pairs still at -1 :", [k for k, v in dist.items() if v == -1])
m = min_dfa(chain)
print("|Q| before %d, after %d" % (len(chain["Q"]), len(m["Q"])))
assert len(m["Q"]) == len(chain["Q"])
print("already minimal.")

A mergeable machine reaches its fixed point in fewer rounds and shrinks.

In [ ]:
redundant = md2mc('''DFA
I  : 0 -> A
I  : 1 -> B
A  : 0 | 1 -> F
B  : 0 | 1 -> F
F  : 0 | 1 -> F
''')
d2, r2 = fixpoint(redundant)
print("rounds :", r2, "  equivalent pairs :", [k for k, v in d2.items() if v == -1])
print("|Q| %d -> %d" % (len(redundant["Q"]), len(min_dfa(redundant)["Q"])))
assert len(min_dfa(redundant)["Q"]) < len(redundant["Q"])

Idempotence: minimizing a minimal machine changes nothing. That *is* the fixed point.

In [ ]:
m = min_dfa(redundant)
mm = min_dfa(m)
print("min(min(D)) isomorphic to min(D)?", iso_dfa(m, mm))
assert iso_dfa(m, mm) and len(m["Q"]) == len(mm["Q"])

## 4. Exercises


1. Why is the bound $|Q|-1$ and not $|Q|$?
2. Name two other fixed-point computations in this book.
3. Construct a DFA needing exactly 5 rounds.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6-DFAOps/Concept-Fixed-Point-Minimization')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')